**Пример бинарной классификации опухолей молочной железы с использованием LightGBM**

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from sales_dataset import SalesDataset

In [3]:
# 1. Загрузка данных
# data = load_breast_cancer()
# X, y = data.data, data.target

df_xls = pd.read_excel('temp_8_9_10_итог_с_июлем.xlsx')
ds = SalesDataset(df_xls, start_date='2024-01-01', cn_mes=19)
X, y = ds.load_data()

# # 2. Разделение данных
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
#
# # 3. Создание датасетов LightGBM
# train_data = lgb.Dataset(X_train, label=y_train)
# test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [4]:
y

array([20., 27., 19., ..., 21., 16., 12.])

In [ ]:
print(f'{type(X)=}, {X.shape=}')
print(f'{type(y)=}, {y.shape=}')

In [ ]:
# 4. Задание параметров модели
params = {
    'objective': 'binary',           # Задача бинарной классификации
    'metric': 'binary_logloss',      # Метрика оценки - логлосс
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'verbose': -1
}

In [ ]:
# 5. Обучение модели
gbm = lgb.train(params,
                train_data,
                num_boost_round=100,
                valid_sets=[test_data],
                callbacks=[lgb.early_stopping(10)])

In [ ]:
print(f'{type(X_test)=}, {X_test.shape=}')

In [ ]:
# 6. Прогнозирование и оценка
# Предсказание вероятностей
y_pred_prob = gbm.predict(X_test, num_iteration=gbm.best_iteration)
# Преобразование вероятностей в метки (порог 0.5)
y_pred = np.where(y_pred_prob > 0.5, 1, 0)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_prob)
print(f'Accuracy: {accuracy:.3f}')
print(f'ROC-AUC: {auc:.3f}')